# 🧭 Phase 5 — Validation Charts (Python)

### 🔍 Context from Previous Phase
In **Phase 4**, we generated rule-based explanations and transaction-level outputs (`valid_reasons_v4.csv`) that combined model scores with plain-language reasons.  
Now, in **Phase 5**, we use Python to validate those results visually, verify performance metrics, and communicate the model’s behavior through clear charts.

---

### 🎯 Objective
Create quick, transparent visualizations to evaluate model quality and explanation coverage.  
All charts will be generated directly from exported artifacts (no Tableau required) and saved under `/artifacts/`.

---

### 📦 What We’ll Produce
| Chart | Purpose |
|--------|----------|
| **1️⃣ Score Distribution** | Check if most transaction scores cluster near 0 as expected (non-fraud majority). |
| **2️⃣ Precision–Recall Curve** | Visualize the global trade-off between catching more frauds (recall) and avoiding false positives (precision). |
| **3️⃣ Threshold vs Precision/Recall** | Identify which decision threshold best fits business risk (e.g., high recall vs high precision). |
| **4️⃣ Top Explanation Reasons** | Summarize the most frequent rule-based reasons that appeared in explanations. |

---

### 🧰 Inputs & Outputs
**Inputs**
- `valid_reasons_v4.csv` → Validation transactions with `fraud_score`, `isFraud`, `reason_text`
- `reason_summary_top10.csv` (optional) → Top reason counts (from Phase 4)

**Outputs (Images & Files)**
- `plot_score_hist_valid.png` → Fraud score distribution  
- `plot_pr_curve.png` → Precision–Recall curve  
- `plot_threshold_vs_pr.png` → Precision/Recall vs thresholds  
- `plot_top_reasons.png` → Top explanation reasons  
- `threshold_sweep_metrics.csv` → numeric threshold metrics  
- `kpi_charts_summary.json` → summary for reports and slides  

---

### 🧱 Next Steps  
- Review the images in `/artifacts/` and embed them into your report or presentation deck to demonstrate model performance and explanation coverage.


In [3]:
# %% [markdown]
# # Phase 5 — Python Validation & KPI Charts

import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.metrics import precision_recall_curve, average_precision_score

# -----------------------------
# Paths
# -----------------------------
PROJ = Path("/Users/animeshchoubey/Downloads/Capstone project")
ART  = PROJ / "artifacts"

valid_csv = ART / "valid_reasons_v4.csv"            # TransactionID, isFraud, fraud_score, reason_text
reasons_csv = ART / "reason_summary_top10.csv"      # optional (we’ll recompute if missing)

assert valid_csv.exists(), f"Missing file: {valid_csv}"

# -----------------------------
# Load data
# -----------------------------
df = pd.read_csv(valid_csv)
# Ensure expected columns present / types right
if df["isFraud"].dtype != np.int64 and df["isFraud"].dtype != np.int32:
    # handle True/False or floats
    df["isFraud"] = df["isFraud"].astype(int)

y_true = df["isFraud"].values
y_score = df["fraud_score"].values

print(f"Loaded {len(df):,} rows | fraud rate: {df['isFraud'].mean():.4f}")

# -----------------------------
# 1) Score distribution
# -----------------------------
plt.figure(figsize=(7,4.2))
plt.hist(y_score, bins=50)
plt.title("Fraud Score Distribution (valid)")
plt.xlabel("fraud_score")
plt.ylabel("Count")
plt.tight_layout()
fig1_path = ART / "plot_score_hist_valid.png"
plt.savefig(fig1_path, dpi=150)
plt.close()
print("Saved:", fig1_path)


Loaded 118,108 rows | fraud rate: 0.0344
Saved: /Users/animeshchoubey/Downloads/Capstone project/artifacts/plot_score_hist_valid.png


In [4]:
# -----------------------------
# 2) Precision–Recall Curve + AP
# -----------------------------
precision, recall, thresh = precision_recall_curve(y_true, y_score)
ap = average_precision_score(y_true, y_score)

plt.figure(figsize=(6.2,5.4))
plt.plot(recall, precision)
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title(f"Precision–Recall Curve (AP = {ap:.3f})")
plt.grid(True, alpha=0.25)
plt.tight_layout()
fig2_path = ART / "plot_pr_curve.png"
plt.savefig(fig2_path, dpi=150)
plt.close()
print("Saved:", fig2_path)

Saved: /Users/animeshchoubey/Downloads/Capstone project/artifacts/plot_pr_curve.png


In [5]:
# -----------------------------
# 3) Threshold → Precision/Recall sweep
#    (and pick best F2 threshold)
# -----------------------------
def prec_rec_at_thresh(y, p, t):
    pred = (p >= t).astype(int)
    tp = np.sum((pred==1) & (y==1))
    fp = np.sum((pred==1) & (y==0))
    fn = np.sum((pred==0) & (y==1))
    precision = tp / (tp + fp) if (tp+fp)>0 else np.nan
    recall    = tp / (tp + fn) if (tp+fn)>0 else np.nan
    return precision, recall, tp, fp, fn

ths = np.linspace(0.01, 0.99, 99)
rows = []
for t in ths:
    P, R, tp, fp, fn = prec_rec_at_thresh(y_true, y_score, t)
    # F2 puts 4x weight on recall
    if (P is np.nan) or (R is np.nan) or (P is None) or (R is None):
        F2 = np.nan
    else:
        beta2 = 2**2
        denom = (beta2*P + R)
        F2 = (1+beta2) * (P*R) / denom if denom>0 else np.nan
    rows.append((t, P, R, F2, tp, fp, fn))

thr_df = pd.DataFrame(rows, columns=["threshold","precision","recall","F2","tp","fp","fn"])
thr_df.to_csv(ART / "threshold_sweep_metrics.csv", index=False)

best = thr_df.loc[thr_df["F2"].idxmax()]
print(f"Best F2 @ threshold={best.threshold:.3f} | P={best.precision:.3f}, R={best.recall:.3f}, F2={best.F2:.3f} | tp={int(best.tp)}, fp={int(best.fp)}, fn={int(best.fn)}")

plt.figure(figsize=(7.2,4.4))
plt.plot(thr_df["threshold"], thr_df["precision"], label="Precision")
plt.plot(thr_df["threshold"], thr_df["recall"],    label="Recall")
plt.axvline(best.threshold, linestyle="--")
plt.text(best.threshold, 0.02, f" t*={best.threshold:.2f}", rotation=90, va="bottom")
plt.ylim(0,1)
plt.xlabel("Threshold")
plt.ylabel("Score")
plt.title("Precision / Recall vs Threshold")
plt.legend()
plt.grid(True, alpha=0.25)
plt.tight_layout()
fig3_path = ART / "plot_threshold_vs_pr.png"
plt.savefig(fig3_path, dpi=150)
plt.close()
print("Saved:", fig3_path)

Best F2 @ threshold=0.100 | P=0.219, R=0.576, F2=0.434 | tp=2341, fp=8345, fn=1723
Saved: /Users/animeshchoubey/Downloads/Capstone project/artifacts/plot_threshold_vs_pr.png


In [6]:
# -----------------------------
# 4) Top Reasons bar chart (robust)
# -----------------------------
def load_or_build_top_reasons(df_valid: pd.DataFrame, reasons_csv_path: Path) -> pd.DataFrame:
    """
    Return a DataFrame with columns ['reason','count'].
    - If reason_summary_top10.csv exists with *any* 2-column shape, coerce/rename.
    - Otherwise recompute from df_valid['reason_text'].
    """
    tr = None
    if reasons_csv_path.exists():
        try:
            tr = pd.read_csv(reasons_csv_path)
            # Normalize column names
            tr.columns = [str(c).strip().lower() for c in tr.columns]
            if tr.shape[1] == 2:
                # Try to coerce to ['reason','count']
                cols = list(tr.columns)
                # If first col isn't 'reason', make it so.
                if 'reason' not in cols:
                    tr.rename(columns={cols[0]: 'reason'}, inplace=True)
                # Ensure the other column is 'count'
                cols = list(tr.columns)
                other = [c for c in cols if c != 'reason'][0]
                if other != 'count':
                    tr.rename(columns={other: 'count'}, inplace=True)
                # minimal sanity
                tr = tr[['reason', 'count']]
            else:
                # Unexpected shape -> force rebuild
                tr = None
        except Exception:
            tr = None

    if tr is None:
        # Recompute from explanations
        s = (
            df_valid['reason_text']
            .astype(str)
            .str.split('; ')
            .explode()
            .str.strip()
            .replace('', np.nan)
            .dropna()
            .value_counts()
            .head(10)
        )
        tr = s.rename_axis('reason').reset_index(name='count')

    # Final cleanups
    tr['reason'] = tr['reason'].astype(str)
    tr['count'] = pd.to_numeric(tr['count'], errors='coerce').fillna(0).astype(int)
    return tr

top_reasons = load_or_build_top_reasons(df, reasons_csv)

plt.figure(figsize=(9, 4.8))
plt.barh(top_reasons['reason'][::-1], top_reasons['count'][::-1])
plt.title("Top Explanation Reasons (valid)")
plt.xlabel("Count")
plt.tight_layout()
fig4_path = ART / "plot_top_reasons.png"
plt.savefig(fig4_path, dpi=150)
plt.close()
print("Saved:", fig4_path)


Saved: /Users/animeshchoubey/Downloads/Capstone project/artifacts/plot_top_reasons.png


In [7]:
# -----------------------------
# 5) Quick text summary (printable block)
# -----------------------------
# ensure all fig paths exist (if a plot was skipped, set placeholder)
fig1_path = ART / "plot_score_hist_valid.png"
fig2_path = ART / "plot_pr_curve.png"
fig3_path = ART / "plot_threshold_vs_pr.png"
fig4_path = ART / "plot_top_reasons.png"

# verify existence or mark "missing"
plots_list = []
for p in [fig1_path, fig2_path, fig3_path, fig4_path]:
    plots_list.append(str(p if p.exists() else f"{p.name} (not generated)"))

summary = {
    "n_valid": int(len(df)),
    "fraud_rate_valid": float(df["isFraud"].mean()),
    "avg_precision_score(AP)": float(ap),
    "best_f2_threshold": float(best.threshold),
    "best_f2_precision": float(best.precision),
    "best_f2_recall": float(best.recall),
    "best_f2": float(best.F2),
    "plots": plots_list,
}

with open(ART / "kpi_charts_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("\n=== SUMMARY ===")
print(json.dumps(summary, indent=2))



=== SUMMARY ===
{
  "n_valid": 118108,
  "fraud_rate_valid": 0.034409184813899145,
  "avg_precision_score(AP)": 0.3684449661339362,
  "best_f2_threshold": 0.09999999999999999,
  "best_f2_precision": 0.21907168257533222,
  "best_f2_recall": 0.5760334645669292,
  "best_f2": 0.4344517853166061,
  "plots": [
    "/Users/animeshchoubey/Downloads/Capstone project/artifacts/plot_score_hist_valid.png",
    "/Users/animeshchoubey/Downloads/Capstone project/artifacts/plot_pr_curve.png",
    "/Users/animeshchoubey/Downloads/Capstone project/artifacts/plot_threshold_vs_pr.png",
    "/Users/animeshchoubey/Downloads/Capstone project/artifacts/plot_top_reasons.png"
  ]
}


In [8]:
import numpy as np, pandas as pd, json
from pathlib import Path

PROJ = Path("/Users/animeshchoubey/Downloads/Capstone project")
ART  = PROJ / "artifacts"
df = pd.read_csv(ART / "valid_reasons_v4.csv")

t = 0.10  # best F2 threshold from summary
y = df["isFraud"].astype(int).values
p = (df["fraud_score"].values >= t).astype(int)

tp = int(((p==1)&(y==1)).sum())
fp = int(((p==1)&(y==0)).sum())
tn = int(((p==0)&(y==0)).sum())
fn = int(((p==0)&(y==1)).sum())

prec = tp / (tp+fp) if (tp+fp)>0 else 0.0
rec  = tp / (tp+fn) if (tp+fn)>0 else 0.0

cm = pd.DataFrame(
    {"actual":[1,0,1,0], "pred":[1,1,0,0], "count":[tp,fp,fn,tn]}
).pivot(index="actual", columns="pred", values="count").rename_axis("Actual").rename(columns={0:"Pred 0",1:"Pred 1"})
cm.to_csv(ART / "confusion_matrix_f2_star.csv")

summary = {
    "threshold": t,
    "TP": tp, "FP": fp, "TN": tn, "FN": fn,
    "precision": round(prec, 4),
    "recall": round(rec, 4)
}
with open(ART / "confusion_f2_star.json", "w") as f: json.dump(summary, f, indent=2)

print(cm)
print("\nSaved:",
      ART / "confusion_matrix_f2_star.csv",
      "and", ART / "confusion_f2_star.json")


pred    Pred 0  Pred 1
Actual                
0       105699    8345
1         1723    2341

Saved: /Users/animeshchoubey/Downloads/Capstone project/artifacts/confusion_matrix_f2_star.csv and /Users/animeshchoubey/Downloads/Capstone project/artifacts/confusion_f2_star.json


In [9]:
# -----------------------------
# Case list export for reviewers
# -----------------------------
import pandas as pd, numpy as np
from pathlib import Path

PROJ = Path("/Users/animeshchoubey/Downloads/Capstone project")
ART  = PROJ / "artifacts"

# Load your Phase 4 explanations
df = pd.read_csv(ART / "valid_reasons_v4.csv")

# --- filter by best threshold ---
THRESHOLD = 0.10  # from F2* summary
case_df = df.loc[df["fraud_score"] >= THRESHOLD].copy()

# --- optional: keep top-N for compact review ---
case_df = case_df.sort_values("fraud_score", ascending=False).head(500)

# --- columns to keep ---
cols = ["TransactionID", "isFraud", "fraud_score", "reason_text"]
case_df = case_df[cols]

out_path = ART / f"case_list_t{THRESHOLD:.2f}.csv"
case_df.to_csv(out_path, index=False)
print(f"✅ Saved case list for reviewers → {out_path}")
print(f"Rows: {len(case_df):,} | Avg fraud_score: {case_df['fraud_score'].mean():.3f}")


✅ Saved case list for reviewers → /Users/animeshchoubey/Downloads/Capstone project/artifacts/case_list_t0.10.csv
Rows: 500 | Avg fraud_score: 0.138


🧭 **Phase 5 — Validation & Model Insights (Summary)**

The Python validation suite confirmed overall model stability and explainability.

Average Precision (AP) = 0.368 at 3.4 % fraud rate.

Best F₂ threshold = 0.10 → Precision ≈ 0.22 | Recall ≈ 0.58 | F₂ ≈ 0.43.

Visual analyses included:

Score Distribution → majority of transactions correctly near 0.

Precision–Recall Curve → smooth trade-off between recall and precision.

Threshold vs Metrics Chart → identified business-optimal cut-off.

Top Reasons Chart → “large entity cluster”, “network connectivity”, and “high amount” as dominant cues.

Artifacts exported: 4 plots + KPI JSON + case-list CSV.

The model is deployment-ready for audit/demo, balancing detection power and human explainability.